In [5]:
import cv2
import numpy as np
from mtcnn import MTCNN

# Inicializamos el detector de caras MTCNN
detector = MTCNN()

# Inicializamos captura de video
video = cv2.VideoCapture(0)

# Definimos la lista de colormaps disponibles
colorMapMethods = [
    cv2.COLORMAP_JET, 
    cv2.COLORMAP_HOT, 
    cv2.COLORMAP_OCEAN, 
    cv2.COLORMAP_RAINBOW, 
    cv2.COLORMAP_DEEPGREEN
]

# Declaramos el índice del colormap actual
i = 0  

# Declaramos un flag para el escaneo
scanning = False  

# Declaramos la variable de progreso del escaneo ( de 0 a 1)
scan_progress = 0  

# Declaramos la velocidad del efecto de escaneo
scan_speed = 0.05  

# Declaramos variable para detectar la primera aparición de la cara
first_detection = True 

# Mostramos los controles en la terminal
print("Controles:")
print("  D - Cambiar colormap (inicia escaneo)")
print("  Q - Salir")
print("\nIniciando...")
print("Esperando detección de rostro...")

# Empezamos el bucle de procesamiento de video
while True:
    
    # Capturamos un frame
    ret, frame = video.read()

    # Si no se captura correctamente, salimos del bucle
    if not ret:
        print("Error al capturar frame")
        break

    # Detectamos caras en el frame
    detections = detector.detect_faces(frame)
    
    # Creamos máscara negra del mismo tamaño que el frame
    mask = np.zeros(frame.shape[:2], dtype=np.uint8)

    # Declaramos una variable para guardar la región facial (bounding box)
    face_region = None
    
    # Si hay caras detectadas, procesamos la primera
    if len(detections) > 0:

        # Guardamos la primera detección
        det = detections[0]  
        x, y, width, height = det['box']
        
        # Si es la primera detección, iniciamos el escaneo
        if first_detection:
            scanning = True
            scan_progress = 0
            first_detection = False
    
        # Guardamos la región facial para el escaneo
        face_region = (x, y, width, height)
        
        # Expandimos la región del rostro para disminuir un poco el error de detección
        padding = int(width * 0.1)  # 10% extra alrededor
        x1 = max(0, x - padding)
        y1 = max(0, y - padding)
        x2 = min(frame.shape[1], x + width + padding)
        y2 = min(frame.shape[0], y + height + padding)
        
        # Calculamos el centro y los ejes de la elipse que cubrirá la cara
        center = ((x1 + x2) // 2, (y1 + y2) // 2)
        axes = ((x2 - x1) // 2, (y2 - y1) // 2)
        
        # La dibujamos en la máscara
        cv2.ellipse(mask, center, axes, 0, 0, 360, 255, -1)
    
    # Aplicamos Gaussian Blur para suavizar los bordes de la máscara
    mask = cv2.GaussianBlur(mask, (51, 51), 0)
    
    # Normalizamos máscara (convertir de 0-255 a 0.0-1.0)
    mask = mask / 255.0

    # Suavizamos frame original
    frameSuavizado = cv2.GaussianBlur(frame, (5, 5), 0)
    
    # Convertimos el frame a escala de grises
    frameGray = cv2.cvtColor(frameSuavizado, cv2.COLOR_BGR2GRAY)
    
    # Convertimos a formato térmico usando el colormap actual
    colorMap = cv2.applyColorMap(frameGray, colorMapMethods[i])

    # Convertir máscara a 3 canales (BGR)
    mask_3ch = np.stack([mask] * 3, axis=2)
    
    # Si estamos en modo escaneo
    if scanning and face_region is not None:

        # Incrementamos el progreso del escaneo
        scan_progress += scan_speed
        
        # Si llegamos al final, terminar escaneo
        if scan_progress >= 1.0:
            scan_progress = 1.0
            scanning = False
        
        # Extraemos coordenadas de la cara
        fx, fy, fw, fh = face_region
        
        # Calculamos hasta dónde ha llegado el escaneo
        scan_y = fy + int(fh * scan_progress)

        # Creamos máscara de escaneo (solo mostrar efecto térmico hasta scan_y)
        scan_mask = np.zeros(frame.shape[:2], dtype=np.float32)
        scan_mask[fy:scan_y, fx:fx+fw] = 1.0  

        # Suavizamos bordes de la máscara de escaneo
        scan_mask = cv2.GaussianBlur(scan_mask, (15, 15), 0)
        scan_mask_3ch = np.stack([scan_mask] * 3, axis=2)
        
        # Combinamos máscara facial con máscara de escaneo
        combined_mask = mask_3ch * scan_mask_3ch
        
        # Aplicamos efecto térmico progresivamente
        resultado = (colorMap * combined_mask + frame * (1 - combined_mask)).astype(np.uint8)
        
        # Dibujamos la Línea de escaneo
        cv2.line(resultado, (fx, scan_y), (fx + fw, scan_y), (255, 255, 255), 3)

        # Dibujamos el texto de escaneo
        cv2.putText(resultado, "SCANNING...", (fx, fy - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        
        # Dibujamos la barra de progreso
        progress_width = int(fw * scan_progress)
        cv2.rectangle(resultado, (fx, fy - 25), (fx + progress_width, fy - 20), (0, 255, 255), -1)
        cv2.rectangle(resultado, (fx, fy - 25), (fx + fw, fy - 20), (0, 255, 255), 2)
    
    else:
        # No hay escaneo: aplicar efecto térmico en toda la máscara
        resultado = (colorMap * mask_3ch + frame * (1 - mask_3ch)).astype(np.uint8)
    
    # Mostramos el resultado
    cv2.imshow('Thermal Vision', resultado)
    
    # Capturamos la tecla presionada
    key = cv2.waitKey(1) & 0xFF
    
    # Cambiar colormap con 'd' e INICIAR ESCANEO
    if key == ord('d'):
        i = (i + 1) % len(colorMapMethods)
        scanning = True  # Activar escaneo
        scan_progress = 0  # Reiniciar progreso
        print(f"Colormap cambiado a: {i} - Iniciando escaneo...")
    
    # Salir con 'q'
    if key == ord('q'):
        break

# Liberar recursos
video.release()
cv2.destroyAllWindows()
print("Programa finalizado")

Controles:
  D - Cambiar colormap (inicia escaneo)
  Q - Salir

Iniciando...
Esperando detección de rostro...
Colormap cambiado a: 1 - Iniciando escaneo...
Colormap cambiado a: 2 - Iniciando escaneo...
Colormap cambiado a: 3 - Iniciando escaneo...
Colormap cambiado a: 4 - Iniciando escaneo...
Colormap cambiado a: 0 - Iniciando escaneo...
Colormap cambiado a: 1 - Iniciando escaneo...
Programa finalizado
